# Silent Zone Detection: Exploratory Data Analysis & Modeling
### AI Decision Support for Emergency Communication Blackouts during Disasters in India
This notebook explores telecommunication infrastructure failure patterns across Floods, Cyclones, and Earthquakes.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project modules can be loaded
sys.path.append('../src')
from data_preprocessing import clean_and_impute_raw_data, validate_dataset_schema
from feature_engineering import engineer_features

df = pd.read_csv('../data/silent_zone_dataset.csv')
print(f"Loaded {len(df)} disaster records with {len(df.columns)} features.")
df.head()

## 1. Class Distribution & Disaster Types

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=df, x='disaster_type', hue='silent_zone', palette='coolwarm', ax=axes[0])
axes[0].set_title('Silent Zone Frequency per Disaster Type')

df['silent_zone'].value_counts().plot.pie(autopct='%1.1f%%', colors=['#5cb85c', '#d9534f'], labels=['Normal Zone', 'Silent Zone'], ax=axes[1])
axes[1].set_ylabel('')
axes[1].set_title('Overall Target Class Distribution')
plt.show()

## 2. Feature Engineering & Correlation Matrix

In [ ]:
df_clean = clean_and_impute_raw_data(df)
df_fe = engineer_features(df_clean)

# Compute correlation with target
corrs = df_fe.select_dtypes(include=[np.number]).corr()['silent_zone'].sort_values(ascending=False)
plt.figure(figsize=(10, 8))
sns.barplot(x=corrs.values, y=corrs.index, palette='viridis')
plt.title('Feature Correlation with Silent Zone Occurrence')
plt.xlabel('Pearson Correlation Coefficient')
plt.show()

## 3. Signal Degradation vs. Tower Operational Status

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df_fe,
    x='tower_operational_percentage',
    y='network_signal_dbm',
    hue='silent_zone',
    palette={0: '#5cb85c', 1: '#d9534f'},
    style='disaster_type',
    s=120
)
plt.axhline(-105, color='gray', linestyle='--', label='Critical Signal Threshold (-105 dBm)')
plt.axvline(30, color='red', linestyle=':', label='Tower Collapse Threshold (30%)')
plt.title('Cellular Signal vs Tower Operational Capacity')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()